# Experiment with definite clauses templates

## Functions

In [ ]:
import torch
import sympy
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict

import sys
sys.path.append('LoH')
from general_models import *
from experiments.generate_data import *
from utils import *


if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')



plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})


In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()
    with torch.no_grad():
        outputs, predicted, labels = predict(model, data_loader, device)
        acc, f1 = eval(predicted, labels)
    model.train()
    return acc, f1

        
def train_model(model, train_loader, optimizer, n_epochs, device, 
                val_loader=None, batches_per_eval=1, verbose=True, early_stopping=False, criterion=torch.nn.BCELoss(), early_stopping_patience=200):
    
    f1_scores_train = []
    f1_scores_val = []  
    model.to(device)
    
    model.train()
    batch = 0
    counter_no_improvement = 0
    last_f1s = [0, 0]
    stopped = False

    for epoch in range(n_epochs):
        if stopped:
            break
        for choose in model.operands:
            choose.gradients = []
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            if verbose:
                for choose in model.operands:
                    choose.gradients.append(choose.Z.grad[0].item())
            optimizer.step()
        
            batch += 1
            if batch % batches_per_eval == 0:
                print(f"Epoch {epoch+1}/{n_epochs}, Batch {batch} - Loss: {loss.item():.6f}")

                acc_train, f1_train = evaluate_model(model, train_loader, device)
                acc_val, f1_val = evaluate_model(model, val_loader, device) if val_loader else (None, None)
                f1_scores_train.append(f1_train)
                f1_scores_val.append(f1_val)

                print(f"\t Train Acc: {acc_train:.2f}, Train F1: {f1_train:.4f}")
                print(f"\t Val Acc: {acc_val:.2f}, Val F1: {f1_val:.4f}")

                if verbose:
                    #print(f"Train Loss: {loss.item():.6f}")
                    formula = model.extract_formula()
                    print("Formula: ", formula)
                    simplified_formula = sympy.parse_expr(formula)
                    print("Simplified formula: ", simplified_formula)

                    print("Weights: ")
                    for i, choose in enumerate(model.operands):
                        print(f"{i}  weight: {choose.get_weights()[0].item():.3f}  gradient: {choose.gradients} \t {choose}")
            
            if f1_val == last_f1s[0] and f1_train == last_f1s[1]:
                counter_no_improvement += 1
            else:
                counter_no_improvement = 0
                last_f1s = [f1_val, f1_train]

            if early_stopping and ((f1_train > 0.999 or f1_val > 0.999) or counter_no_improvement > early_stopping_patience):
                print("Early stopping at epoch ", epoch)
                stopped = True
                break
            
    return f1_scores_train, f1_scores_val


def select_rules_architecture2(rules_strs, proposition_names):
    parser = LogicalParser(proposition_names)
    architecture = " | ".join([f"[{rule}, ~{(rule)}, False]" for rule in rules_strs])
    return parser.parse(architecture)


def experiment(n_rules, proposition_names, lr=0.15, n_epochs=15, repetitions=20, verbose=False, baches_per_eval=1, early_stopping=True, criterion=torch.nn.BCELoss(),
               fixed_length_model=False, choose_subset_rules=False, horn=True, heads_given=False, first_clause=False):
    
    global train_loader, test_loader
    global device
    
    f1_train = []
    f1_val = []

    negated_propositions = [f"~{p}" for p in proposition_names]
    
    if first_clause:
        n_rules = n_rules - 1

    for i in range(repetitions):
        print(f"Repetition {i+1}/{repetitions}")
        
        rules = []
        if not horn:
            for _ in range(n_rules):
                rules.append(select_rules_architecture2(proposition_names, proposition_names))
        else:
            if not heads_given:
                select_heads = [select_one_rule_architecture(proposition_names, proposition_names, choosedual=False) for _ in range(n_rules)]
            else:
                select_heads = heads_given
            if not fixed_length_model:
                for head in select_heads:
                    rules.append(Or(select_rules_architecture(negated_propositions, proposition_names, conjunction=False, choosedual=False), head))
            else:
                list_oracles = [negated_propositions for _ in range(fixed_length_model)]
                for head in select_heads:
                    rules.append(Or(select_one_rule_per_list_architecture(list_oracles, proposition_names, conjunction=False, choosedual=False), head))
        
        if choose_subset_rules:
            model = select_rules_architecture(rules, proposition_names, conjunction=True, choosedual=True).to(device)
        else:
            model = And(*rules).to(device)
        if first_clause:
            model = And(model, first_clause).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        print("Model: ", model)
        print()
        f1_scores_train, f1_scores_val = train_model(model, train_loader, optimizer, n_epochs, device, val_loader=test_loader, 
                                                     verbose=verbose, early_stopping=early_stopping, batches_per_eval=baches_per_eval, criterion=criterion)
        f1_train.append(f1_scores_train)
        f1_val.append(f1_scores_val)
        print()
        print(model.extract_formula())
        print("----------------------------------------------------------")
        print()
        
    # fill in the missing values with last value
    max_length = max(len(f1) for f1 in f1_train)
    for i in range(repetitions):
        while len(f1_train[i]) < max_length:
            f1_train[i].append(f1_train[i][-1])
            f1_val[i].append(f1_val[i][-1])

    f1_train = np.array(f1_train)
    f1_val = np.array(f1_val)
        
    return model, f1_train, f1_val


## Comparisons

In [ ]:
num_clauses = 5
num_literals = 10
clause_length_range = (2, 2)
proposition_names = [f"P_{i+1}" for i in range(num_literals)]

random_cnf = [[-3, -8, 7], [-10, -3, 4], [-1, -9, 10], [-2, -6, 8], [-4, -3, 5]]
named_list = [["~P_3", "~P_8", "P_7"], ["~P_10", "~P_3", "P_4"], ["~P_1", "~P_9", "P_10"], ["~P_2", "~P_6", "P_8"], ["~P_4", "~P_3", "P_5"]]
named_strs = ["(~P_3 | ~P_8 | P_7)", "(~P_10 | ~P_3 | P_4)", "(~P_1 | ~P_9 | P_10)", "(~P_2 | ~P_6 | P_8)", "(~P_4 | ~P_3 | P_5)"]
named_str = "(~P_3 | ~P_8 | P_7) & (~P_10 | ~P_3 | P_4) & (~P_1 | ~P_9 | P_10) & (~P_2 | ~P_6 | P_8) & (~P_4 | ~P_3 | P_5)"

# Generate all truth assignments
assignments = generate_all_assignments(num_literals)

# Evaluate formula and produce dataset labels
results = evaluate_formula(assignments, random_cnf, formula_type='cnf')

#  Split the dataset into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(assignments + 0., results + 0., test_size=0.25, random_state=42)
print(X_train.shape, y_train.shape)

# Data loaders
batch_size = 128
train_dataset =  TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


repetitions = 20
n_epochs = 64
batches_per_eval = 1
lr = 0.15
experiment_models_fixed = {}
print("Solution: \t \t ", named_str)
print(f"Proportion true labels:   {100*torch.sum(results).item() / len(results):.1f} %")
print()

In [ ]:
num_rules_model = num_clauses
fixed_length_model = False
choose_subset_rules = False
horn = False
experiment_models_fixed["5 clauses"]  = experiment(num_rules_model, proposition_names, fixed_length_model = fixed_length_model, 
                                            choose_subset_rules = choose_subset_rules, horn = horn, 
                                            lr=lr, n_epochs=n_epochs, repetitions=repetitions, 
                                            verbose=False, baches_per_eval=batches_per_eval, early_stopping=True)

In [ ]:
num_rules_model = num_clauses
fixed_length_model = False
choose_subset_rules = False
horn = True
experiment_models_fixed["5 definite clauses"]  = experiment(num_rules_model, proposition_names, fixed_length_model = fixed_length_model, 
                                            choose_subset_rules = choose_subset_rules, horn = horn,
                                            lr=lr, n_epochs=n_epochs, repetitions=repetitions, 
                                            verbose=False, baches_per_eval=batches_per_eval, early_stopping=True)

In [ ]:
num_rules_model = num_clauses
fixed_length_model = 2
choose_subset_rules = False
horn = True
experiment_models_fixed["5 definite clauses of width 3"]  = experiment(num_rules_model, proposition_names, fixed_length_model = fixed_length_model, 
                                            choose_subset_rules = choose_subset_rules, horn = horn,
                                            lr=lr, n_epochs=n_epochs, repetitions=repetitions, 
                                            verbose=False, baches_per_eval=batches_per_eval, early_stopping=True)

In [ ]:
num_rules_model = num_clauses
fixed_length_model = False
choose_subset_rules = False
horn = True
heads = [Prop(h-1, f"P_{h}") for h in list(zip(*random_cnf))[-1]]
print("Heads given: ", heads)
experiment_models_fixed["5 definite clauses with heads given"]  = experiment(num_rules_model, proposition_names, fixed_length_model = fixed_length_model, 
                                            choose_subset_rules = choose_subset_rules, horn = horn,
                                            lr=lr, n_epochs=n_epochs, repetitions=repetitions, 
                                            verbose=False, baches_per_eval=batches_per_eval, early_stopping=True, heads_given=heads)

In [ ]:
num_rules_model = num_clauses
fixed_length_model = False
choose_subset_rules = False
horn = True
parser = LogicalParser(proposition_names)
first_clause = parser.parse(named_strs[0])
experiment_models_fixed["5 definite clauses with first given"]  = experiment(num_rules_model, proposition_names, fixed_length_model = fixed_length_model, 
                                            choose_subset_rules = choose_subset_rules, horn = horn,
                                            lr=lr, n_epochs=n_epochs, repetitions=repetitions, 
                                            verbose=False, baches_per_eval=batches_per_eval, early_stopping=True, first_clause=first_clause)

In [ ]:
num_rules_model = num_clauses
# plot f1 scores mean with std
print(f"num_true_clauses={num_clauses}, num_literals={num_literals}, clause_length_range={clause_length_range} \n"
            + f"proportion_true_labels={100*torch.sum(results).item() / len(results):.1f}%, num_rules_model={num_rules_model} \n" 
            + f"repetitions={repetitions}, batch_size={batch_size}, lr={lr}")
for model_name in experiment_models_fixed.keys():
    (model, f1_train, f1_val) = experiment_models_fixed[model_name]
    plt.plot(f1_val.mean(axis=0), label=model_name)
#plt.title(f"num_true_clauses={num_clauses}, num_literals={num_literals}, clause_length_range={clause_length_range} \n"
#            + f"proportion_true_labels={100*torch.sum(results).item() / len(results):.1f}%, num_new_clauses={len(additional_rules_)} \n" 
#            + f"repetitions={repetitions}, batch_size={batch_size}, lr={lr}")
#plt.title("Comparison of different models, with a fixed target set of clauses")
plt.ylim(0.32, 1.02)
plt.xlabel(f'Steps')
plt.ylabel('F1 score')
#plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
# Save the dictionary
import pickle
with open('LoH/experiments/results/definite_clauses20.pickle', 'wb') as f:
    pickle.dump(experiment_models_fixed, f)

In [ ]:
# Load the dictionary
with open('LoH/experiments/results/definite_clauses20.pickle', 'rb') as f:
    experiment_models_fixed = pickle.load(f)